Ce code permet de créer le panel final qui nous servira pour les estimations.
À partir des fichiers panel_va_STAN.csv & panel_cit1.csv, on procède à plusieurs modifications :
 - création des secteurs agrégés C & H
 - création du vecteur d'intensité de R&D
 - création des agrégats de R&D : intra-sectoriel, extra-sectoriel, domestique et étranger
 - ajout d'une DUMMY pour la crise de 2009
 - calcul de l'indicateur HHI (Herfindahl-Hirschmann) par secteur
 - séparation en 2 panels distincts selon l'intensité de R&D moyenne pour chaque pays, pour chaque base de données

Ce code retourne des panels longs pour chaque base prêt à être utilisé pour les estimations.

Version 2.0 : lit le panel du run choisi dans `project_settings.py` (RD_UNIT, DIMENSION, TECH), lit l'intensité `intensite_propre.csv` (RDV / PRODV, `prépa_va_STAN`) et écrit les panels dans `runs/<run>/panels/`.

In [109]:
import numpy as np
import pandas as pd
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_WANTED, METRIC_SPE, YEARS, RESULTS_DIR, RUN_DIR, PANEL_CODE_DIR, CHOC, CHOC_SCENARIO
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")
import os

CHOC = False -> D:/ENTICE/rendu_fin_stage/runs/SRDQ (c,s)/


In [110]:
panel_file = RESULTS_DIR + f"panel_estimation/panel_estimation_{BASE}.csv"
print("Panel amont lu :", panel_file)
df = pd.read_csv(panel_file)

# La nouvelle mesure doit déjà être présente dans le CSV produit par calc_blocs.
if BASE == "CAPITAL" and METRIC_WANTED == "embod_investment":
    investment_metric = METRIC_SPE["CAPITAL_embod_investment"]
    investment_columns = [
        f"rd_same_sector_same_country_{investment_metric}",
        f"rd_same_sector_diff_country_{investment_metric}",
        f"rd_diff_sector_same_country_{investment_metric}",
        f"rd_diff_sector_diff_country_{investment_metric}",
    ]
    missing = [c for c in investment_columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"Colonnes investment absentes du panel lu : {missing}. "
            "Relancer calc_blocs modifié avec BASE='CAPITAL' et "
            "METRIC_WANTED='embod_investment', en vérifiant le même RESULTS_DIR. "
            f"Fichier attendu : {panel_file}"
        )


Panel amont lu : D:/ENTICE/rendu_fin_stage/runs/SRDQ (c,s)/panel_estimation/panel_estimation_CAPITAL.csv


In [111]:
print(df.columns)

Index(['Region', 'Sector', 'year',
       'rd_same_sector_same_country_rd_dir_CAPITAL',
       'rd_same_sector_diff_country_rd_dir_CAPITAL',
       'rd_diff_sector_same_country_rd_dir_CAPITAL',
       'rd_diff_sector_diff_country_rd_dir_CAPITAL',
       'rd_same_sector_same_country_rd_embod_CAPITAL',
       'rd_same_sector_diff_country_rd_embod_CAPITAL',
       'rd_diff_sector_same_country_rd_embod_CAPITAL',
       'rd_diff_sector_diff_country_rd_embod_CAPITAL',
       'rd_same_sector_same_country_rd_embod_investment_CAPITAL',
       'rd_same_sector_diff_country_rd_embod_investment_CAPITAL',
       'rd_diff_sector_same_country_rd_embod_investment_CAPITAL',
       'rd_diff_sector_diff_country_rd_embod_investment_CAPITAL', 'exports',
       'imports_final_demand', 'imports_intermediate', 'production'],
      dtype='str')


In [112]:
df = df.rename(columns={"year": "annee"})

Création des secteurs agrégés

In [113]:
c_subsectors = df.loc[df['Sector'].str.startswith('C', na=False), 'Sector'].unique().tolist()

mask_c = df['Sector'].isin(c_subsectors)

phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

if BASE in phys:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',

            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',

            'exports',
            'imports_final_demand',
            'imports_intermediate',
            'production']
else:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}']

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    col += ["rd_same_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_same_sector_diff_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL",]

c_agg = (
    df[mask_c]
    .groupby(['Region', 'annee'], as_index=False)
    [col]
    .sum()
)
c_agg['Sector'] = 'C'

df_final = pd.concat([df, c_agg], ignore_index=True)

In [114]:
if BASE in phys:
    h_subsectors = df.loc[df['Sector'].str.startswith('H', na=False), 'Sector'].unique().tolist()
    mask_h = df_final['Sector'].isin(h_subsectors)

    h_agg = (
        df_final[mask_h]
        .groupby(['Region', 'annee'], as_index=False)
        [col]
        .sum()
    )
    h_agg['Sector'] = 'H'

    df_final1 = pd.concat([df_final, h_agg], ignore_index=True)
else:
    df_final1 = df_final

In [115]:
os.makedirs(RESULTS_DIR + 'panels', exist_ok=True)
df_final1.to_csv(RESULTS_DIR + f'panels/panel_{BASE}_intermediaire_F.csv')

In [116]:
df_final1

,Region,Sector,annee,rd_same_sector_same_country_rd_dir_CAPITAL,rd_same_sector_diff_country_rd_dir_CAPITAL,rd_diff_sector_same_country_rd_dir_CAPITAL,rd_diff_sector_diff_country_rd_dir_CAPITAL,rd_same_sector_same_country_rd_embod_CAPITAL,rd_same_sector_diff_country_rd_embod_CAPITAL,rd_diff_sector_same_country_rd_embod_CAPITAL,rd_diff_sector_diff_country_rd_embod_CAPITAL,rd_same_sector_same_country_rd_embod_investment_CAPITAL,rd_same_sector_diff_country_rd_embod_investment_CAPITAL,rd_diff_sector_same_country_rd_embod_investment_CAPITAL,rd_diff_sector_diff_country_rd_embod_investment_CAPITAL,exports,imports_final_demand,imports_intermediate,production
0,AT,A,1995,0.000003,0.000003,0.003966,0.001589,0.055752,0.079344,178.246731,72.936890,0.052945,0.069602,146.089335,51.390687,181.963870,1.486011e+04,908.345668,8.794762e+03
1,AT,B,1995,0.000000,0.000000,0.016953,0.002086,0.015815,0.033874,35.945044,11.073996,0.014241,0.024686,31.026642,8.135760,56.363108,1.486011e+04,100.182006,1.136038e+03
2,AT,C10_C12,1995,0.000000,0.000000,0.005370,0.000422,0.229348,0.058853,101.391449,26.506048,0.207355,0.049656,88.469296,18.928471,587.010739,1.486011e+04,930.614479,1.005106e+04
3,AT,C13_C15,1995,0.000000,0.000000,0.012247,0.000472,0.017832,0.039208,34.050204,6.603548,0.016237,0.034436,30.653251,4.667865,922.116132,1.486011e+04,163.445058,1.737351e+03
4,AT,C16_C18,1995,0.000000,0.000000,0.006926,0.001334,0.732992,0.114209,123.096624,41.273798,0.652557,0.093567,108.594533,31.866611,1935.597272,1.486011e+04,973.615961,1.098599e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39307,US,H,2016,0.022661,0.000022,0.213381,0.017887,22660.578499,527.687601,221575.010040,32392.145193,17345.752830,285.023922,189495.689503,24336.630392,91115.016860,5.189148e+06,52352.957520,2.787177e+06
39308,US,H,2017,0.022803,0.000024,0.227147,0.017136,25795.134055,601.343572,249692.541170,33919.234543,19130.141028,317.285527,210277.093309,24886.156361,93306.015761,5.414875e+06,52651.401301,3.005142e+06
39309,US,H,2018,0.023422,0.000022,0.206812,0.015131,26876.462293,584.735893,226927.793957,29592.708170,20671.197805,306.182112,194292.890970,21989.139951,94730.399206,5.471909e+06,61229.908341,3.103204e+06
39310,US,H,2019,0.020419,0.000020,0.223487,0.016537,27734.156647,672.872825,267338.241694,36053.691795,21245.224839,352.023728,229011.912699,26863.422255,103810.094507,6.024783e+06,67532.491401,3.367028e+06


Découpe des différents panels

In [117]:
if BASE in phys:
    va_STAN = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/Panels_intermediaires/panel_va_STAN.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]

    df_final1 = df_final1.rename(columns={"production": f"production_{BASE}"})
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])
else:
    va_STAN = pd.read_csv(PATH_WRK + 'code/création_panel_estimation/Panels_intermediaires/panel_cit1.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])

df

,Region,Sector,annee,VA_V,VA_Q,VA_costFactors,comp_emp,emp,net_stock_capital,production,...,rd_diff_sector_same_country_rd_embod_CAPITAL,rd_diff_sector_diff_country_rd_embod_CAPITAL,rd_same_sector_same_country_rd_embod_investment_CAPITAL,rd_same_sector_diff_country_rd_embod_investment_CAPITAL,rd_diff_sector_same_country_rd_embod_investment_CAPITAL,rd_diff_sector_diff_country_rd_embod_investment_CAPITAL,exports,imports_final_demand,imports_intermediate,production_CAPITAL
0,AT,A,1995,3843.823732,3410.753099,5023.693732,348.500000,38.626784,3.428939e+04,7664.729833,...,178.246731,72.936890,0.052945,0.069602,146.089335,51.390687,181.963870,1.486011e+04,908.345668,8.794762e+03
1,AT,A,1996,3588.173927,3354.544562,4309.291927,354.200000,38.047075,3.438198e+04,7436.299892,...,79.251222,62.333406,0.030086,0.064775,66.373874,45.453975,214.838322,1.727854e+04,880.684899,8.202025e+03
2,AT,A,1997,3693.656888,3478.446066,4251.222888,364.700000,38.474740,3.479585e+04,7714.325937,...,94.771978,64.780299,0.032909,0.062839,75.446807,47.166225,236.581217,1.936133e+04,932.554859,8.348596e+03
3,AT,A,1998,3590.057497,3625.242586,4249.225497,366.000000,37.816498,3.498068e+04,7467.688729,...,179.737374,64.377388,0.062204,0.087665,148.408301,46.659706,295.848279,2.087913e+04,919.797470,8.043761e+03
4,AT,A,1999,3616.638040,3843.736884,4315.897040,382.300000,39.327879,3.516822e+04,7480.581422,...,106.657970,60.634926,0.063820,0.250034,83.701332,44.123349,310.963809,2.235032e+04,959.229835,8.198995e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14149,US,R+S+T,2015,403824.172649,433197.497956,393560.340571,267416.971634,10318.000000,1.022369e+06,670507.180879,...,7279.359980,1211.738083,0.438873,0.100713,4895.216874,687.799785,27650.469925,1.033579e+06,14280.929361,9.130389e+05
14150,US,R+S+T,2016,420530.767816,438314.944498,409734.759237,278968.564174,10471.000000,1.047762e+06,701452.164244,...,7986.011661,1243.450292,0.524754,0.110007,5392.126134,713.577505,27263.110241,1.037830e+06,13930.696062,9.776044e+05
14151,US,R+S+T,2017,435249.731220,449286.366872,423538.087025,289233.014358,10519.000000,1.062726e+06,727146.899842,...,9201.956079,1357.317562,0.553530,0.131811,6085.966697,767.499949,25652.049924,1.082975e+06,14401.154481,9.668593e+05
14152,US,R+S+T,2018,457138.704255,465631.493211,444723.251620,304570.694274,10767.000000,1.082054e+06,763918.279636,...,9321.639206,1357.103857,0.592629,0.139841,6305.315957,781.572567,27400.950825,1.094382e+06,15193.418535,9.958279e+05


In [118]:
# embod
if BASE in phys:
    col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_F'] = df[col_F].sum(axis=1)
    df['rd_embod_D'] = df[col_D].sum(axis=1)

    col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_I'] = df[col_I].sum(axis=1)
    df['rd_embod_X'] = df[col_X].sum(axis=1)

    col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod'] = df[col_S].sum(axis=1)

#dir
col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_F'] = df[col_F].sum(axis=1)
df['rd_dir_D'] = df[col_D].sum(axis=1)

col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_I'] = df[col_I].sum(axis=1)
df['rd_dir_X'] = df[col_X].sum(axis=1)

col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir'] = df[col_S].sum(axis=1)

df["2009_crisis"]=(df["annee"]==2009).astype(int)

# col = 'production_EXIOBASE_mapF'
# df['share'] = df['production'] / df.groupby(['Sector', 'annee'])[col].transform('sum')
df['share'] = df['production'] / df.groupby(['Sector', 'annee'])['production'].transform('sum')
df['HHI'] = df.groupby(['Sector', 'annee'])['share'].transform(lambda s: (s**2).sum())

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    same_same = "rd_same_sector_same_country_rd_embod_investment_CAPITAL"
    same_diff = "rd_same_sector_diff_country_rd_embod_investment_CAPITAL"
    diff_same = "rd_diff_sector_same_country_rd_embod_investment_CAPITAL"
    diff_diff = "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL"

    df["rd_embod_investment_D"] = df[[same_same, diff_same]].sum(axis=1)
    df["rd_embod_investment_F"] = df[[same_diff, diff_diff]].sum(axis=1)
    df["rd_embod_investment_I"] = df[[same_same, same_diff]].sum(axis=1)
    df["rd_embod_investment_X"] = df[[diff_same, diff_diff]].sum(axis=1)
    df["rd_embod_investment"] = df[[same_same, same_diff, diff_same, diff_diff]].sum(axis=1)

In [119]:
df = df.drop(columns=['share'])

On ajoute l'intensité en R&D

In [120]:
df['intensité'] = df[f'rd_{METRIC_WANTED}'] / df[f'production']

In [121]:
df.to_csv(RESULTS_DIR + f'panels/panel_{BASE}_F.csv', index=False)

Division des panels par base en deux groupes de pays en fonction de leur intensité de R&D moyenne

In [122]:
# R&D propre / production : fichier RDV / PRODV calculé une seule fois par prépa_va_STAN (indépendant de la base et de la technologie)
# rd_propre = flux de R&D en M€ courants, production = production STAN aux prix courants en M€ (taux de change de l'année), C et H inclus
df_int = (pd.read_csv(PANEL_CODE_DIR + "Panels_intermediaires/intensite_propre.csv")
            .rename(columns={"rd_propre_V": "rd_propre", "production_V": "production"}))

# vraie intensité = R&D propre du secteur / production du secteur
df_int["intensité_vraie"] = df_int["rd_propre"] / df_int["production"]

# intensité annuelle propre R/Y (en points de %), pour le terme croisé de l'estimation
df_int.loc[df_int["production"] <= 1.0, "intensité_vraie"] = np.nan   # production quasi nulle => intensité absurde
int_annual = (df_int[["Region", "Sector", "annee", "intensité_vraie"]]
              .rename(columns={"intensité_vraie": "int_propre"}))
int_annual["int_propre"] = 100 * int_annual["int_propre"]


In [123]:
# Panel complet, avec int_propre
df = df.merge(int_annual, on=["Region", "Sector", "annee"], how="left")

os.makedirs(RESULTS_DIR + "panels", exist_ok=True)

df.to_csv(RESULTS_DIR + f"panels/panel_{BASE}_F.csv", index=False)
